In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc

DIR_PROCESADOS = Path("../data/processed")

tcel = sc.read_h5ad(DIR_PROCESADOS / "04_linfocitos_T.h5ad")

pb = sc.get.aggregate(tcel, by="muestra", func="sum", layer="counts")
conteos = pd.DataFrame(np.asarray(pb.layers["sum"]).T,
                       index=pb.var_names, columns=pb.obs_names).round().astype(int)
conteos.shape

(21761, 16)

In [2]:
meta = (tcel.obs.groupby("muestra", observed=True)
        .agg(condicion=("condicion", "first"),
             donante=("donante", "first"),
             n_T=("condicion", "size")))
meta = meta[meta["n_T"] >= 20]
conteos = conteos[meta.index]

print(meta.sort_values("condicion"))
print(meta["condicion"].value_counts())


          condicion donante  n_T
muestra                         
486_LS           LS     486   35
BIO023_LS        LS  BIO023   23
DER114_LS        LS  DER114  298
DER115_LS        LS  DER115  432
DER119_LS        LS  DER119   73
DER114_NL        NL  DER114  101
DER115_NL        NL  DER115  200
DER119_NL        NL  DER119   52
DER120_NL        NL  DER120   30
465            Sano     465   29
632            Sano     632   33
CON161         Sano  CON161   27
CON171         Sano  CON171   71
CON194         Sano  CON194   32
condicion
LS      5
Sano    5
NL      4
Name: count, dtype: int64


In [3]:
conteos.to_csv(DIR_PROCESADOS / "pseudobulk_T_conteos.csv")
meta.to_csv(DIR_PROCESADOS / "pseudobulk_T_meta.csv")